# LoRA del mio volto · FLUX.1 [dev] su Kaggle T4 (kohya sd-scripts)

**Uso:** solo personale/portfolio (FLUX.1-dev ha licenza **non commerciale**).
Le foto restano nel tuo dataset Kaggle privato e in `/tmp`; su Hugging Face va **solo il LoRA**, in un repository **privato** (il LoRA "contiene" il tuo volto: non renderlo pubblico).

**Prima di iniziare:**
1. Su Hugging Face, da loggato, accetta la licenza di `black-forest-labs/FLUX.1-dev` (*Agree and access repository*).
2. Crea un token HF con permesso **Write**. Su Kaggle: Add-ons → Secrets → `HF_TOKEN`, **collegato a questo notebook**.
3. Session options → **Accelerator: GPU T4 x2** · **Internet: On**.
4. Add Input → il tuo **dataset privato** con le 20–30 foto.

**Come si usa (due passaggi):**
1. **Pilota:** `MODE = "pilot"` → *Run All* in modalità interattiva (~30–45 min, ~17 GB di download inclusi). Alla fine la cella 7 stampa il verdetto: velocità reale, presenza di NaN e durata stimata dell'addestramento completo.
2. **Completo:** imposta `MODE = "full"` e `MIXED_PRECISION` come indicato dal verdetto, poi **Save Version → Save & Run All (Commit)**. Gira in background fino a 12 h: puoi chiudere il browser. Ogni `SAVE_EVERY` passi un LoRA intermedio viene caricato su HF, così non perdi nulla se la sessione si interrompe.

> ⚠ VERIFICA = punto in cui nomi di repository/file o opzioni potrebbero essere cambiati: controlla le model card su HF e `docs/flux_train_network.md` di kohya-ss/sd-scripts.

## 0 · Configurazione
L'unica cella da modificare. `TRIGGER` è la parola chiave che "attiva" il tuo volto: dovrai scriverla in ogni prompt di generazione.

In [ ]:
# ==== WHAT TO RUN ====
MODE = "pilot"   # "pilot": short test, run it interactively first
                 # "full" : full training, run it with Save Version -> Save & Run All (Commit)

# ==== IDENTITY ====
TRIGGER   = "mgcnv man"        # rare token + class word: put it in every generation prompt
LORA_NAME = "mario-flux-lora"  # file name and Hugging Face repo name
HF_REPO   = None               # None = "<your HF user>/<LORA_NAME>" (always private)

# ==== PATHS ====
INPUT_DIR = "/kaggle/input"          # your private dataset with the photos
WORK_DIR  = "/tmp/lora"              # models, dataset copy, caches: NOT saved in the notebook output
OUT_DIR   = "/kaggle/working/lora"   # trained LoRA files + logs (saved in the output)

# ==== MODELS ====  ⚠ VERIFY on the model cards
FLUX = ("Kijai/flux-fp8", "flux1-dev-fp8-e4m3fn.safetensors")    # FLUX.1-dev already in fp8: half the download, no slow CPU cast
# FLUX = ("black-forest-labs/FLUX.1-dev", "flux1-dev.safetensors")  # official 23.8 GB: cast to fp8 at every start (slow, more RAM)
AE   = ("black-forest-labs/FLUX.1-dev", "ae.safetensors")         # gated: needs the accepted license
CLIP = ("comfyanonymous/flux_text_encoders", "clip_l.safetensors")
T5   = ("comfyanonymous/flux_text_encoders", "t5xxl_fp8_e4m3fn.safetensors")  # NOT the _scaled variant (unsupported)
SD_SCRIPTS_REF = "main"   # kohya-ss/sd-scripts branch or tag

# ==== TRAINING ====
RESOLUTION      = 768      # 512 = faster, 1024 = sharper face but too slow on T4
MAX_STEPS       = 1600     # ~60 steps per photo (25 photos -> 1500)
SAVE_EVERY      = 400      # intermediate LoRA every N steps, uploaded to HF during training
NETWORK_DIM     = 16
NETWORK_ALPHA   = 16
LEARNING_RATE   = 1e-4
MIXED_PRECISION = "fp16"   # set it from the pilot verdict: "fp16" or "bf16"
BLOCKS_TO_SWAP  = 18       # transformer blocks kept in RAM instead of VRAM: raise it on "CUDA out of memory"
SEED            = 42

# ==== PILOT ====
PILOT_STEPS    = 50
PILOT_TRY_BF16 = True      # if fp16 gives NaN, repeat the pilot in bf16 (emulated on T4: slower)

## 1 · Controllo GPU, RAM e disco
Si ferma se la GPU non è adatta (es. P100). La T4 (Turing, capability 7.5) non ha bf16 nativo: si prova prima **fp16**.

In [ ]:
import sys, os, platform, shutil, subprocess, time, json, glob, re
import torch, psutil

print("Python :", platform.python_version())
print("Torch  :", torch.__version__, "| CUDA:", torch.version.cuda)
assert torch.cuda.is_available(), "No GPU. Session options -> Accelerator -> GPU T4 x2"
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"GPU {i}  : {p.name} | capability {p.major}.{p.minor} | {p.total_memory/1e9:.1f} GB")

cap = torch.cuda.get_device_capability(0)
if cap < (7, 5):
    raise SystemExit(f"GPU capability {cap} is too old for this training (e.g. P100). Choose GPU T4 x2.")
if torch.cuda.get_device_properties(0).total_memory < 14e9:
    raise SystemExit("Less than 14 GB of VRAM: FLUX training will not fit. Choose GPU T4 x2.")
print("bf16   :", "native" if cap[0] >= 8 else "not native (Turing): fp16 first, bf16 only emulated")

os.makedirs(WORK_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)
ram = psutil.virtual_memory().total / 1e9
print(f"RAM    : {ram:.1f} GB")
print(f"Disk   : {shutil.disk_usage(WORK_DIR).free/1e9:.1f} GB free in {WORK_DIR} | "
      f"{shutil.disk_usage(OUT_DIR).free/1e9:.1f} GB free in {OUT_DIR}")
if ram < 24:
    print("⚠ Less than 24 GB of RAM: block swap may crash the session. Use Kaggle GPU T4 x2.")
print("Training uses GPU 0 only (kohya trains on a single GPU).")

## 2 · Installazione di kohya sd-scripts (torch bloccato)
- Clona `kohya-ss/sd-scripts` in `/tmp` e installa i suoi `requirements.txt`.
- Un file di *constraints* blocca torch/torchvision alla versione di Kaggle: nessun pacchetto può cambiarli.
- Alla fine verifica, in un processo separato, che i moduli Flux di kohya e `bitsandbytes` si importino.

In [ ]:
import importlib.metadata as md_

REPO_DIR = f"{WORK_DIR}/sd-scripts"
TORCH_BEFORE = md_.version("torch")
CONSTRAINTS = "/tmp/constraints.txt"
pins = [f"torch=={TORCH_BEFORE}"]
for pkg in ("torchvision", "torchaudio"):
    try:
        pins.append(f"{pkg}=={md_.version(pkg)}")
    except md_.PackageNotFoundError:
        pass
with open(CONSTRAINTS, "w") as f:
    f.write("\n".join(pins) + "\n")

if not os.path.exists(f"{REPO_DIR}/flux_train_network.py"):
    subprocess.check_call(["git", "clone", "--depth", "1", "--branch", SD_SCRIPTS_REF,
                           "https://github.com/kohya-ss/sd-scripts", REPO_DIR])
print("sd-scripts:", subprocess.check_output(["git", "-C", REPO_DIR, "log", "-1", "--format=%h %cs %s"], text=True).strip())

# requirements.txt ends with "-e .": it must run from the repo folder
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-c", CONSTRAINTS, "-r", "requirements.txt"],
                      cwd=REPO_DIR)

if md_.version("torch") != TORCH_BEFORE:
    print(f"⚠ torch changed {TORCH_BEFORE} -> {md_.version('torch')}: Run -> Restart session, then rerun.")
check = subprocess.run([sys.executable, "-c", "import library.flux_utils, networks.lora_flux, bitsandbytes, accelerate; print('ok')"],
                       cwd=REPO_DIR, capture_output=True, text=True)
for pkg in ["accelerate", "transformers", "diffusers", "bitsandbytes", "huggingface_hub"]:
    print(f"  {pkg}=={md_.version(pkg)}")
if check.returncode != 0:
    print(check.stderr[-2000:])
    raise SystemExit("❌ kohya modules do not import: do NOT continue. See 'Errori comuni' at the end.")
print("✅ sd-scripts ready")

## 3 · Login a Hugging Face e repository privato
Legge `HF_TOKEN` dai Secrets (il valore non viene mai stampato), verifica la licenza di FLUX.1-dev e crea il repository **privato** per il LoRA. Se il repository esiste già ed è pubblico, si ferma.

In [ ]:
from huggingface_hub import login, whoami, HfApi, hf_hub_download, hf_hub_url, get_hf_file_metadata

def get_secret(name):
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        pass
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        pass
    return os.environ[name]

login(token=get_secret("HF_TOKEN"), add_to_git_credential=False)
api = HfApi()
user = whoami()["name"]
print("HF user:", user)

try:
    get_hf_file_metadata(hf_hub_url(*AE))
except Exception as e:
    raise SystemExit(f"❌ No access to {AE[0]} ({type(e).__name__}): accept the license at "
                     f"https://huggingface.co/{AE[0]} with the SAME account as the token.")
print("FLUX.1-dev license: OK")

REPO_ID = HF_REPO or f"{user}/{LORA_NAME}"
api.create_repo(REPO_ID, private=True, exist_ok=True)
if not api.model_info(REPO_ID).private:
    raise SystemExit(f"❌ {REPO_ID} exists and is PUBLIC: make it private on HF or change LORA_NAME.")
print("LoRA repo (private): https://huggingface.co/" + REPO_ID)

## 4 · Download dei modelli (~17 GB, in `/tmp`)
FLUX.1-dev fp8 (~12 GB) + VAE + CLIP-L + T5-XXL fp8. Controlla prima lo spazio libero. Se rieseguita nella stessa sessione non riscarica nulla.

In [ ]:
MODELS_DIR = f"{WORK_DIR}/models"
sizes = {(repo, fn): get_hf_file_metadata(hf_hub_url(repo, fn)).size or 0 for repo, fn in (FLUX, AE, CLIP, T5)}
missing = sum(sz for (_, fn), sz in sizes.items() if not os.path.exists(f"{MODELS_DIR}/{fn}"))
free = shutil.disk_usage(WORK_DIR).free
print(f"To download: {missing/1e9:.1f} GB | free: {free/1e9:.1f} GB")
if missing > free - 5e9:
    raise SystemExit("❌ Not enough disk: keep at least 5 GB free for caches and checkpoints.")

PATHS = {}
for (repo, fn), sz in sizes.items():
    t0 = time.time()
    PATHS[fn] = hf_hub_download(repo, fn, local_dir=MODELS_DIR)
    print(f"  {fn:40s} {sz/1e9:5.1f} GB  {time.time()-t0:5.0f}s")
FLUX_PATH, AE_PATH, CLIP_PATH, T5_PATH = (PATHS[m[1]] for m in (FLUX, AE, CLIP, T5))

## 5 · Preparazione delle foto
- Corregge la rotazione EXIF e salva copie **senza metadati** (niente GPS) in `/tmp`, ridimensionate a max 1536 px.
- Didascalie: se nel dataset accanto a `foto.jpg` c'è `foto.txt`, usa quella (aggiungendo il `TRIGGER` se manca); altrimenti `photo of <TRIGGER>`. Per un solo soggetto su FLUX funziona bene anche così.
- Mostra il provino: scarta (togliendole dal dataset) le foto sfocate, con occhiali da sole o con altre persone.

In [ ]:
from PIL import Image, ImageOps
import matplotlib.pyplot as plt

EXTS = (".jpg", ".jpeg", ".png", ".webp", ".jfif")
everything = glob.glob(f"{INPUT_DIR}/**/*", recursive=True)
srcs = sorted(p for p in everything if p.lower().endswith(EXTS))
heic = [p for p in everything if p.lower().endswith((".heic", ".heif"))]
if heic:
    print(f"⚠ {len(heic)} HEIC photos skipped: export them as JPG and update the dataset")
assert srcs, f"No photos in {INPUT_DIR}: Add Input -> your private dataset"

DATA_DIR = f"{WORK_DIR}/dataset/img"
shutil.rmtree(f"{WORK_DIR}/dataset", ignore_errors=True)   # old caches too: they are rebuilt for these photos
os.makedirs(DATA_DIR)

small, rows, from_txt = [], [], 0
for i, p in enumerate(srcs, 1):
    im = ImageOps.exif_transpose(Image.open(p)).convert("RGB")   # new image: EXIF (GPS included) is dropped
    w, h = im.size
    if min(w, h) < RESOLUTION:
        small.append(f"{os.path.basename(p)} ({w}x{h})")
    s = min(1.0, 1536 / max(w, h))
    if s < 1:
        im = im.resize((round(w * s), round(h * s)), Image.LANCZOS)
    name = f"img_{i:03d}"
    im.save(f"{DATA_DIR}/{name}.jpg", quality=95)

    txt = os.path.splitext(p)[0] + ".txt"
    if os.path.exists(txt):
        cap = open(txt, encoding="utf-8").read().strip()
        from_txt += 1
    else:
        cap = f"photo of {TRIGGER}"
    if TRIGGER not in cap:
        cap = f"{TRIGGER}, {cap}"
    with open(f"{DATA_DIR}/{name}.txt", "w", encoding="utf-8") as f:
        f.write(cap)
    rows.append((name, im, cap))

n = len(rows)
print(f"{n} photos | {from_txt} captions from .txt | example caption: {rows[0][2]!r}")
if n < 15:
    print("⚠ Fewer than 15 photos: the likeness will be weaker. 20-30 is ideal.")
if n > 40:
    print("⚠ More than 40 photos: keep the best 25-30, more only slows training.")
if small:
    print(f"⚠ {len(small)} photos smaller than {RESOLUTION}px on the short side (they lower the sharpness):")
    for s_ in small:
        print("   ", s_)

cols = 8
fig, ax = plt.subplots((n + cols - 1) // cols, cols, figsize=(2.2 * cols, 2.6 * ((n + cols - 1) // cols)), squeeze=False)
for a in ax.flat:
    a.axis("off")
for a, (name, im, _) in zip(ax.flat, rows):
    a.imshow(im); a.set_title(f"{name}\n{im.size[0]}x{im.size[1]}", fontsize=8)
plt.tight_layout(); plt.show()

## 6 · Configurazione dell'addestramento e funzioni
- `dataset.toml`: risoluzione `RESOLUTION` con *bucket* (le foto non quadrate non vengono deformate), batch 1, niente flip orizzontale (il volto non è simmetrico).
- Comando kohya come nella documentazione ufficiale per GPU da 16 GB: FLUX in **fp8**, `blocks_to_swap`, gradient checkpointing, AdamW 8 bit, didascalie e latenti pre-calcolati.
- `train(...)` mostra una riga di stato al minuto, **si ferma da sola se la loss diventa NaN** (per non sprecare quota) e, nel run completo, carica su HF ogni LoRA intermedio appena salvato.

In [ ]:
import select
from concurrent.futures import ThreadPoolExecutor
from safetensors.torch import load_file

DATASET_TOML = f"{WORK_DIR}/dataset.toml"
with open(DATASET_TOML, "w") as f:
    f.write(f"""[general]
caption_extension = ".txt"
shuffle_caption = false
flip_aug = false

[[datasets]]
resolution = {RESOLUTION}
batch_size = 1
enable_bucket = true
bucket_no_upscale = true

  [[datasets.subsets]]
  image_dir = "{DATA_DIR}"
  num_repeats = 1
""")

def build_cmd(steps, precision, out_dir, name, save_every=None):
    # ⚠ VERIFY the options in docs/flux_train_network.md of sd-scripts
    cmd = [sys.executable, "-m", "accelerate.commands.launch",
           "--num_processes", "1", "--num_machines", "1", "--mixed_precision", precision,
           "--dynamo_backend", "no", "--num_cpu_threads_per_process", "2",
           "flux_train_network.py",
           "--pretrained_model_name_or_path", FLUX_PATH, "--clip_l", CLIP_PATH, "--t5xxl", T5_PATH, "--ae", AE_PATH,
           "--dataset_config", DATASET_TOML, "--output_dir", out_dir, "--output_name", name,
           "--save_model_as", "safetensors", "--save_precision", "fp16",
           "--network_module", "networks.lora_flux",
           "--network_dim", str(NETWORK_DIM), "--network_alpha", str(NETWORK_ALPHA),
           "--network_train_unet_only",
           "--optimizer_type", "AdamW8bit", "--learning_rate", str(LEARNING_RATE), "--lr_scheduler", "constant",
           "--mixed_precision", precision, "--fp8_base", "--sdpa", "--gradient_checkpointing",
           "--blocks_to_swap", str(BLOCKS_TO_SWAP),
           "--guidance_scale", "1.0", "--timestep_sampling", "shift", "--discrete_flow_shift", "3.1582",
           "--model_prediction_type", "raw",
           "--cache_text_encoder_outputs", "--cache_text_encoder_outputs_to_disk",
           "--cache_latents", "--cache_latents_to_disk",
           "--max_train_steps", str(steps), "--seed", str(SEED),
           "--max_data_loader_n_workers", "2", "--persistent_data_loader_workers"]
    if save_every:
        cmd += ["--save_every_n_steps", str(save_every)]
    return cmd

class Uploader:
    """Uploads every finished .safetensors in `folder` to the private HF repo, in a background thread."""
    def __init__(self, repo_id, folder):
        self.repo_id, self.folder, self.done, self.futs = repo_id, folder, set(), []
        self.pool = ThreadPoolExecutor(1)

    def scan(self, final=False):
        for p in sorted(glob.glob(f"{self.folder}/*.safetensors")):
            if p in self.done or (not final and time.time() - os.path.getmtime(p) < 30):  # < 30 s: still being written
                continue
            self.done.add(p)
            self.futs.append(self.pool.submit(self._upload, p))

    def _upload(self, p):
        api.upload_file(path_or_fileobj=p, path_in_repo=os.path.basename(p), repo_id=self.repo_id,
                        commit_message=f"Add {os.path.basename(p)}")
        print(f"  ↑ {os.path.basename(p)} uploaded to HF")

    def finish(self):
        self.scan(final=True)
        for f in self.futs:
            f.result()   # re-raise upload errors here

STEP_RE = re.compile(r"steps:\s*\d+%.*?(\d+)/(\d+)")
LOSS_RE = re.compile(r"avr_loss=([^\s,\]]+)")

def fmt_h(seconds):
    return f"{int(seconds // 3600)}h{int(seconds % 3600 // 60):02d}m"

def train(steps, precision, out_dir, name, save_every=None, upload_to=None, tag="run"):
    os.makedirs(out_dir, exist_ok=True)
    log_path = f"{out_dir}/{tag}.log"
    env = {**os.environ, "CUDA_VISIBLE_DEVICES": "0", "PYTHONUNBUFFERED": "1"}
    print(f"▶ {tag}: {steps} steps | {precision} | {RESOLUTION}px | log: {log_path}")
    t_start = time.time()
    proc = subprocess.Popen(build_cmd(steps, precision, out_dir, name, save_every), cwd=REPO_DIR, env=env,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    st = dict(step=0, loss=None, t_first=None, step_first=None, t_last=None, nan=False)
    uploader = Uploader(upload_to, out_dir) if upload_to else None
    buf, last_print, last_scan = b"", 0.0, time.time()

    def s_per_it():
        if st["t_first"] is None or st["step"] <= st["step_first"]:
            return None
        return (st["t_last"] - st["t_first"]) / (st["step"] - st["step_first"])

    with open(log_path, "wb") as log:
        while True:
            ready, _, _ = select.select([proc.stdout], [], [], 30)
            if ready:
                chunk = os.read(proc.stdout.fileno(), 65536)
                if not chunk:
                    break   # EOF: the process has ended
                log.write(chunk); log.flush()
                buf += chunk
                parts = re.split(rb"[\r\n]", buf)
                buf = parts.pop()
                for raw in parts:
                    line = raw.decode("utf-8", "replace").strip()
                    if not line:
                        continue
                    m = STEP_RE.search(line)
                    if not m:
                        if "%|" not in line:      # skip the other progress bars (caching, loading)
                            print(line[:300])
                        continue
                    now, step = time.time(), int(m[1])
                    if step > st["step"]:
                        if st["t_first"] is None and step >= 2:   # step 1 includes the warm-up
                            st["t_first"], st["step_first"] = now, step
                        st["step"], st["t_last"] = step, now
                    lm = LOSS_RE.search(line)
                    if lm:
                        st["loss"] = lm[1]
                        if re.search(r"nan|inf", lm[1], re.I):
                            st["nan"] = True
                    if st["nan"]:
                        print(f"❌ loss = {st['loss']} at step {st['step']}: stopping ({precision} is unstable here)")
                        proc.terminate()
                        break
                    if now - last_print > 60 or step == steps:
                        spi = s_per_it()
                        eta = f" | ETA {fmt_h((steps - step) * spi)}" if spi else ""
                        print(f"  step {step}/{steps} | {spi or 0:.1f} s/step | loss {st['loss']} | "
                              f"elapsed {fmt_h(now - t_start)}{eta}")
                        last_print = now
                if st["nan"]:
                    break
            if uploader and time.time() - last_scan > 60:
                uploader.scan()
                last_scan = time.time()

    rc = proc.wait()
    if uploader:
        uploader.finish()
    res = dict(tag=tag, precision=precision, rc=rc, nan=st["nan"], steps_done=st["step"], loss=st["loss"],
               s_per_it=s_per_it(), setup_s=(st["t_first"] or time.time()) - t_start,
               elapsed_s=time.time() - t_start, log=log_path, lora=f"{out_dir}/{name}.safetensors")
    if rc != 0 and not st["nan"]:
        print(f"❌ exit code {rc}. Last lines of the log:")
        tail = open(log_path, "rb").read()[-6000:].decode("utf-8", "replace")
        print("\n".join(l for l in re.split(r"[\r\n]", tail) if l.strip())[-3000:])
    return res

def check_lora(path):
    if not os.path.exists(path):
        print(f"❌ {os.path.basename(path)} not found")
        return False
    sd = load_file(path)
    bad = [k for k, v in sd.items() if not torch.isfinite(v.float()).all()]
    print(f"{os.path.basename(path)}: {len(sd)} tensors, {os.path.getsize(path)/1e6:.0f} MB, "
          + (f"❌ NaN/Inf in {len(bad)} tensors" if bad else "✅ all values finite"))
    return not bad

## 7 · Pilota (solo con `MODE = "pilot"`)
Addestra `PILOT_STEPS` passi in **fp16**; se compaiono NaN e `PILOT_TRY_BF16 = True`, ripete in **bf16**. Poi stampa il **verdetto**: precisione da usare, secondi per passo e durata stimata del run completo, confrontata con il limite di 12 h di Kaggle.
Il LoRA del pilota resta in `/tmp` e non viene caricato su HF.

In [ ]:
PILOT = []
if MODE != "pilot":
    print(f"MODE = {MODE!r}: pilot skipped")
else:
    precisions = [MIXED_PRECISION] + (["bf16"] if PILOT_TRY_BF16 and MIXED_PRECISION == "fp16" else [])
    for prec in precisions:
        r = train(PILOT_STEPS, prec, f"{WORK_DIR}/pilot_{prec}", "pilot", tag=f"pilot_{prec}")
        r["ok"] = r["rc"] == 0 and not r["nan"] and check_lora(r["lora"])
        PILOT.append(r)
        if r["ok"] or not r["nan"]:   # bf16 only helps against NaN, not against crashes or OOM
            break

    print("\n" + "=" * 70 + "\nVERDICT")
    good = [r for r in PILOT if r["ok"]]
    if not good:
        for r in PILOT:
            print(f"  {r['precision']}: " + ("NaN" if r["nan"] else f"failed (exit code {r['rc']})") + f" -> {r['log']}")
        print("❌ FLUX training does not work on this T4 setup.\n"
              "   - 'CUDA out of memory' in the log: raise BLOCKS_TO_SWAP (e.g. 22-25) or set RESOLUTION = 512, then retry.\n"
              "   - NaN in fp16 and bf16: go to Plan B (SDXL LoRA), which is stable in fp16 on T4.")
    else:
        r = good[0]
        spi = r["s_per_it"]
        eta = r["setup_s"] + MAX_STEPS * spi
        print(f"✅ {r['precision']} works: {spi:.1f} s/step, setup {r['setup_s']/60:.0f} min (loading + caching)")
        print(f"   Full run: {MAX_STEPS} steps @ {RESOLUTION}px ≈ {fmt_h(eta)} "
              f"(Kaggle: max 12 h per run, ~30 h of GPU per week)")
        if eta > 11 * 3600:
            fit = int((11 * 3600 - r["setup_s"]) / spi)
            print(f"⚠ Too long for one run. Options: MAX_STEPS = {fit // 100 * 100} (with SAVE_EVERY = 300), "
                  f"or RESOLUTION = 512 (roughly 2x faster), then rerun the pilot.")
        print(f"\nNext: set MIXED_PRECISION = {r['precision']!r} and MODE = \"full\" in cell 0, "
              f"then Save Version -> Save & Run All (Commit).")
    with open(f"{OUT_DIR}/pilot_result.json", "w") as f:
        json.dump([{k: v for k, v in r.items()} for r in PILOT], f, indent=2)

## 8 · Addestramento completo (solo con `MODE = "full"`)
Da lanciare con **Save Version → Save & Run All (Commit)**: gira in background e il log si legge nella pagina della versione.
I LoRA intermedi (`<nome>-step00000400.safetensors`, …) e quello finale (`<nome>.safetensors`) finiscono in `/kaggle/working/lora` e, appena salvati, nel tuo repository privato su HF.

In [ ]:
FULL = None
if MODE != "full":
    print(f"MODE = {MODE!r}: full training skipped")
else:
    FULL = train(MAX_STEPS, MIXED_PRECISION, OUT_DIR, LORA_NAME, save_every=SAVE_EVERY, upload_to=REPO_ID, tag="full")
    print(f"\nDone in {fmt_h(FULL['elapsed_s'])} | exit code {FULL['rc']} | last loss {FULL['loss']}")

## 9 · Verifica e scheda del modello
Controlla che ogni LoRA salvato non contenga NaN e carica su HF un `README.md` con trigger e parametri (resta privato).
**Quale LoRA usare:** di solito il finale. Se nelle prove di generazione il volto sembra "incollato" o rigido (overfitting), prova l'intermedio precedente.

In [ ]:
if MODE == "full" and FULL is not None:
    loras = sorted(glob.glob(f"{OUT_DIR}/*.safetensors"))
    for p in loras:
        check_lora(p)
    card = f"""---
base_model: black-forest-labs/FLUX.1-dev
license: other
license_name: flux-1-dev-non-commercial-license
tags: [flux, lora, personal]
instance_prompt: {TRIGGER}
---
# {LORA_NAME}

Personal LoRA for FLUX.1 [dev], trained with kohya-ss/sd-scripts on a Kaggle T4. **Keep this repository private.**

- Trigger: `{TRIGGER}` (e.g. "photo of {TRIGGER}, ...")
- Photos: {n} · resolution {RESOLUTION} · steps {FULL['steps_done']}/{MAX_STEPS} · rank {NETWORK_DIM} / alpha {NETWORK_ALPHA}
- lr {LEARNING_RATE} · AdamW8bit · {MIXED_PRECISION} · fp8 base · timestep shift 3.1582
- Files: {", ".join(os.path.basename(p) for p in loras)}
"""
    api.upload_file(path_or_fileobj=card.encode(), path_in_repo="README.md", repo_id=REPO_ID,
                    commit_message="Add model card")
    print("Model card uploaded: https://huggingface.co/" + REPO_ID)
else:
    print("Nothing to check (MODE is not 'full' or training did not run).")

---
## Errori comuni

| Errore | Causa | Soluzione |
|---|---|---|
| `401 Unauthorized` | token errato o revocato | rigenera il token su HF e aggiorna il secret |
| `403` / `GatedRepoError` / "No access to black-forest-labs/FLUX.1-dev" | licenza non accettata, o token di un altro account | accetta la licenza con lo **stesso** account del token |
| `403` su `create_repo` / `upload_file` | token in sola lettura | crea un token con permesso **Write** |
| `SecretNotFound` | secret non collegato | Add-ons → Secrets → spunta `HF_TOKEN` |
| `ConnectionError` | Internet spento | Session options → Internet On (richiede verifica del telefono) |
| `No photos in /kaggle/input` | dataset non aggiunto | Add Input → il tuo dataset privato |
| errore di pip nella cella 2 | un requisito di kohya è incompatibile con il torch di Kaggle | leggi il messaggio: di solito basta togliere il pin indicato da `requirements.txt` o usare un `SD_SCRIPTS_REF` precedente |
| `kohya modules do not import` | dipendenza mancante o versione sbagliata | leggi l'errore stampato; se cita `bitsandbytes`, `pip install -U bitsandbytes` |
| `CUDA out of memory` | VRAM insufficiente | `BLOCKS_TO_SWAP` più alto (22–25) oppure `RESOLUTION = 512` |
| la sessione si riavvia / "kernel died" durante il caricamento | RAM esaurita | usa il checkpoint fp8 di `FLUX` (default) e non il file ufficiale da 23,8 GB |
| `loss = nan` | instabilità di fp16 su FLUX | il pilota prova bf16 da solo; se fallisce anche quello → Piano B (SDXL) |
| `Unsupported fp8 model dtype` | checkpoint fp8 di tipo e5m2 o "scaled" | usa `flux1-dev-fp8-e4m3fn` e `t5xxl_fp8_e4m3fn` (non `_scaled`) |
| run completo interrotto a 12 h | troppi passi per la velocità della T4 | riparti dall'ultimo LoRA intermedio su HF, oppure riduci `MAX_STEPS` / `RESOLUTION` |
| quota GPU esaurita | ~30 h/settimana su Kaggle (⚠ VERIFICA) | il pilota consuma poco; lancia il completo solo dopo un verdetto ✅ |